In [139]:
import pandas as pd
import numpy as np
from ollama import chat
import ollama
import json
import torch
from torch.nn.functional import cosine_similarity

In [45]:
df = pd.read_json("../data/all_courses_data.json")

In [46]:
df = df[df["Education cycle"] == "Second-cycle"]

In [47]:
owners = filter(lambda x: x.startswith("MP") or x == "TRACKS", df.Owner.unique())
df = df[df.Owner.isin(owners)]

In [ ]:
df

In [61]:
df["Aim"][931]

'The aim of the course is to give a good theoretical foundation for some of the most important areas of optimisation: convex optimisation, linear optimisation, and nonlinear optimisation. The course treats the basics principles for analyzing the properties of an optimisation problem, and the characterization of feasible points that are (locally) optimal. This theory is used to develop a number of examples of optimisation methods, that can be used to solve instances of practical optimisation problem. The course also aims to give some practical training in mathematical modelling and problem solving using optimisation.'

In [ ]:
df["Learning outcomes"][931]

'After completion of the course, the student shall be able to: 1.Demonstrate knowledge and understanding of important concepts and practical issues of financial accounting and management accounting and control. 2. Demonstrate the ability to produce basic group accounts and explain the purposes of these as well as the ideas underlying group accounting. 3. Demonstrate the connection between organization\x92s profitability, financing and growth, related to different types of risk. 3. Demonstrate the ability to identify and formulate both problems and solutions in the area of management accounting and control.'

In [ ]:
df["Content"][931]

'The course comprises two modules: 1. Financial Accounting 2. Management Accounting and Control Module 1 focuses on regulation of financial accounting, basic concepts (e.g. relevance and faithful representation), and group accounting. The section on regulation deals with what rules there are, why there is regulation and the costs and benefits that regulation brings to the economy. Various regulation methods are dealt with, and the rules companies must follow today. The basic concepts section focuses on what underlying structures there are in accounting and how these structures are mirrored in a number of current accounting rules. The course also has a technically oriented section with a special focus on group accounting. In this section, students learn the format of group accounting and the financial concepts behind the methods for producing group accounts. Module 2 deals with various perspectives of management accounting and control and is divided into two parts. The first part consid

In [111]:
60, 43

(60, 43)

In [152]:
aim, outcomes, content = df["Aim"][931], df["Learning outcomes"][931], df["Content"][931]
prompt = f"You are an assistant. Summarise the course to a short summary of at most 200 words (not a strict condition) and only mention things about the course, do not inlcude any unnecessary information. Do not output the number of words. This summary will then be used for retrieval and semantic search: \n\nAIM:\n{aim}\n\nLEARNING_OUTCOMES:\n\n{outcomes}\n\nCONTENT:\n\n{content}"

response = chat(
  model='qwen3:4b',
  messages=[{'role': 'user', 'content': prompt}],
  format='json',
  stream=False,
)
summary_non_lin = json.loads(response.message.content)["summary"]

In [ ]:
aim, outcomes, content = df["Aim"][60], df["Learning outcomes"][60], df["Content"][60]
prompt = f"You are an assistant. Summarise the course to a short summary of at most 200 words (not a strict condition) and only mention things about the course, do not inlcude any unnecessary information. Do not output the number of words. This summary will then be used for retrieval and semantic search: \n\nAIM:\n{aim}\n\nLEARNING_OUTCOMES:\n\n{outcomes}\n\nCONTENT:\n\n{content}"

response = chat(
  model='qwen3:4b',
  messages=[{'role': 'user', 'content': prompt}],
  format='json',
  stream=False,
)

summary_applied_ml = json.loads(response.message.content)["summary"]

In [154]:
summary_applied_ml

'This course provides an introduction to machine learning techniques and theory, emphasizing practical application. Students gain knowledge of common ML problem types, their limitations, and the critical importance of high-quality data and features. They develop skills in implementing algorithms (including linear and nonlinear models), evaluating system performance, and comparing model effectiveness. The course also focuses on practical judgment: analyzing model advantages/limitations, selecting appropriate features, choosing evaluation methods, and addressing ethical implications. Content covers supervised learning (regression, classification, neural networks), unsupervised learning (clustering), and real-world contexts like e-commerce, business intelligence, NLP, image processing, and bioinformatics. It stresses feature engineering, system reliability, and ethical considerations in deploying ML solutions.'

This course provides an introduction to machine learning techniques and theory, emphasizing practical application. Students gain knowledge of common ML problem types, their limitations, and the critical importance of high-quality data and features. They develop skills in implementing algorithms (including linear and nonlinear models), evaluating system performance, and comparing model effectiveness. The course also focuses on practical judgment: analyzing model advantages/limitations, selecting appropriate features, choosing evaluation methods, and addressing ethical implications. Content covers supervised learning (regression, classification, neural networks), unsupervised learning (clustering), and real-world contexts like e-commerce, business intelligence, NLP, image processing, and bioinformatics. It stresses feature engineering, system reliability, and ethical considerations in deploying ML solutions.

In [155]:
aim, outcomes, content = df["Aim"][43], df["Learning outcomes"][43], df["Content"][43]
prompt = f"You are an assistant. Summarise the course to a short summary of at most 200 words (not a strict condition) and only mention things about the course, do not inlcude any unnecessary information. Do not output the number of words. This summary will then be used for retrieval and semantic search: \n\nAIM:\n{aim}\n\nLEARNING_OUTCOMES:\n\n{outcomes}\n\nCONTENT:\n\n{content}"

response = chat(
  model='qwen3:4b',
  messages=[{'role': 'user', 'content': prompt}],
  format='json',
  stream=False,
)
summary_algo_ml = json.loads(response.message.content)["summary"]

In [156]:
summary_algo_ml

'This course covers foundational machine learning algorithms and inference techniques from an AI perspective. It focuses on learning models that generalize from data and inference systems that derive predictions or actions. Key applications include classification tasks (e.g., credit scoring, character recognition), expert systems (e.g., medical diagnosis), and data mining for discovering patterns in large datasets. The curriculum covers supervised learning methods (Bayes classifiers, perceptrons, SVMs, K-nearest neighbors, regression, logistic regression), unsupervised learning approaches (clustering, EM algorithm, mixture models), model selection, kernel methods, and deep learning architectures (fully connected neural networks, CNNs, RNNs). Students gain the ability to implement, analyze, and evaluate these algorithms, apply mathematical principles to hypothesis inference, choose appropriate methods for specific problems, and critically assess their strengths and limitations based on 

This course covers foundational machine learning algorithms and inference techniques from an AI perspective. It focuses on learning models that generalize from data and inference systems that derive predictions or actions. Key applications include classification tasks (e.g., credit scoring, character recognition), expert systems (e.g., medical diagnosis), and data mining for discovering patterns in large datasets. The curriculum covers supervised learning methods (Bayes classifiers, perceptrons, SVMs, K-nearest neighbors, regression, logistic regression), unsupervised learning approaches (clustering, EM algorithm, mixture models), model selection, kernel methods, and deep learning architectures (fully connected neural networks, CNNs, RNNs). Students gain the ability to implement, analyze, and evaluate these algorithms, apply mathematical principles to hypothesis inference, choose appropriate methods for specific problems, and critically assess their strengths and limitations based on scientific literature for practical implementation and evaluation.

In [157]:
embeds = ollama.embed(
  model='qwen3-embedding:4b',
  input=[summary_non_lin, summary_applied_ml, summary_algo_ml]
)

In [158]:
embeds = np.array(embeds["embeddings"])
embeds = torch.from_numpy(embeds)

In [175]:
cosine_similarity(embeds, embeds.roll(1, 0))

tensor([0.4940, 0.4611, 0.7100], dtype=torch.float64)